In [14]:
"""
Cálculo Numérico — Equações Diferenciais
=========================================
Implemente os seguintes métodos utilizando a linguagem Python.

Conteúdo:
  1. Equação Diferencial Ordinária (EDO)
  2. Equação Diferencial Parcial (EDP)
  3. Ordem de Equações Diferenciais
  4. Linearidade de Equações Diferenciais
  5. Método de Euler (reta tangente)
  6. Método de Heun (Euler modificado)
  7. Método de Runge-Kutta (genérico)
  8. Método de Runge-Kutta de 3ª Ordem
  9. Método de Runge-Kutta de 4ª Ordem
"""

import math
import numpy as np
import matplotlib.pyplot as plt
from typing import Callable, List, Tuple, Optional


# ─────────────────────────────────────────────────────────────────────────────
# 1. EQUAÇÃO DIFERENCIAL ORDINÁRIA (EDO)
# ─────────────────────────────────────────────────────────────────────────────

def edo_exemplo(f: Callable[[float, float], float],
                x0: float, y0: float,
                x_final: float, h: float,
                metodo: str = "euler") -> Tuple[List[float], List[float]]:
    """
    Resolve uma Equação Diferencial Ordinária (EDO) de 1ª ordem:
        dy/dx = f(x, y),  y(x0) = y0

    Uma EDO envolve uma função de UMA variável independente e suas derivadas.

    Parâmetros:
        f        : Função f(x, y) que define dy/dx
        x0       : Valor inicial de x
        y0       : Condição inicial y(x0)
        x_final  : Limite superior do intervalo
        h        : Tamanho do passo
        metodo   : 'euler', 'heun' ou 'rk4'

    Retorna:
        (xs, ys) : Listas dos valores de x e y calculados
    """
    metodos_disponiveis = {"euler", "heun", "rk4"}
    if metodo not in metodos_disponiveis:
        raise ValueError(f"Método '{metodo}' inválido. Escolha: {metodos_disponiveis}")

    xs, ys = [x0], [y0]
    x, y = x0, y0

    while x < x_final - 1e-10:
        if metodo == "euler":
            y = euler_passo(f, x, y, h)
        elif metodo == "heun":
            y = heun_passo(f, x, y, h)
        elif metodo == "rk4":
            y = rk4_passo(f, x, y, h)

        x = round(x + h, 10)
        xs.append(x)
        ys.append(y)

    print(f"[EDO] Método '{metodo}' | Passos: {len(xs)-1} | h = {h}")
    return xs, ys


# ─────────────────────────────────────────────────────────────────────────────
# 2. EQUAÇÃO DIFERENCIAL PARCIAL (EDP)
# ─────────────────────────────────────────────────────────────────────────────

def edp_equacao_calor(L: float, T: float,
                      nx: int, nt: int,
                      alpha: float,
                      u_inicial: Callable[[np.ndarray], np.ndarray],
                      u_contorno_esq: float = 0.0,
                      u_contorno_dir: float = 0.0) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Resolve a EDP da Equação do Calor (difusão):
        ∂u/∂t = α · ∂²u/∂x²

    Uma EDP envolve derivadas parciais em DUAS ou mais variáveis independentes.
    Usa o método explícito de diferenças finitas (FTCS).

    Parâmetros:
        L             : Comprimento do domínio espacial [0, L]
        T             : Tempo total de simulação
        nx            : Número de pontos espaciais
        nt            : Número de passos de tempo
        alpha         : Difusividade térmica (α > 0)
        u_inicial     : Função que recebe array x e retorna condição inicial u(x, 0)
        u_contorno_esq: Temperatura na borda esquerda (x=0)
        u_contorno_dir: Temperatura na borda direita (x=L)

    Retorna:
        (x, t, u) : Grades espacial, temporal e solução u[it, ix]

    Estabilidade:
        r = α·Δt/Δx² deve ser ≤ 0.5 para estabilidade numérica.
    """
    dx = L / (nx - 1)
    dt = T / nt
    r = alpha * dt / dx**2

    if r > 0.5:
        print(f"⚠  Aviso: r = {r:.4f} > 0.5 — solução pode ser instável!")
    else:
        print(f"[EDP] Equação do Calor | r = {r:.4f} (estável) | nx={nx}, nt={nt}")

    x = np.linspace(0, L, nx)
    t = np.linspace(0, T, nt + 1)
    u = np.zeros((nt + 1, nx))

    # Condição inicial
    u[0, :] = u_inicial(x)

    # Condições de contorno
    u[:, 0] = u_contorno_esq
    u[:, -1] = u_contorno_dir

    # Avanço no tempo (diferenças finitas explícitas)
    for n in range(nt):
        u[n+1, 1:-1] = u[n, 1:-1] + r * (u[n, 2:] - 2*u[n, 1:-1] + u[n, :-2])

    return x, t, u


# ─────────────────────────────────────────────────────────────────────────────
# 3. ORDEM DE EQUAÇÕES DIFERENCIAIS
# ─────────────────────────────────────────────────────────────────────────────

def verificar_ordem(descricao: str, ordem: int) -> None:
    """
    Exibe e explica a ordem de uma equação diferencial.

    A ORDEM é determinada pela maior derivada presente na equação.

    Exemplos conceituais:
        dy/dx = f(x, y)           → Ordem 1
        d²y/dx² + y = 0           → Ordem 2 (ex: oscilador harmônico)
        d³y/dx³ = g(x)            → Ordem 3

    Parâmetros:
        descricao : Descrição textual da equação
        ordem     : Ordem da equação diferencial
    """
    print(f"\n{'─'*55}")
    print(f"  Equação : {descricao}")
    print(f"  Ordem   : {ordem}ª")
    if ordem == 1:
        print("  Tipo    : Primeira ordem — envolve dy/dx")
    elif ordem == 2:
        print("  Tipo    : Segunda ordem  — envolve d²y/dx²")
    else:
        print(f"  Tipo    : {ordem}ª ordem — maior derivada é d^{ordem}y/dx^{ordem}")
    print(f"{'─'*55}")


def sistema_de_segunda_ordem(f: Callable[[float, float, float], float],
                              t0: float, y0: float, dy0: float,
                              t_final: float, h: float) -> Tuple[List, List, List]:
    """
    Resolve EDO de 2ª ordem convertendo em sistema de 1ª ordem:
        y'' = f(t, y, y')
    com condições iniciais y(t0)=y0 e y'(t0)=dy0.

    Conversão:
        v = y'  →  v' = f(t, y, v)
        sistema: y' = v,  v' = f(t, y, v)

    Parâmetros:
        f       : Função f(t, y, y') que define y''
        t0      : Tempo inicial
        y0      : Condição inicial y(t0)
        dy0     : Condição inicial y'(t0)
        t_final : Tempo final
        h       : Tamanho do passo

    Retorna:
        (ts, ys, vs) : Tempos, posições e velocidades
    """
    ts, ys, vs = [t0], [y0], [dy0]
    t, y, v = t0, y0, dy0

    while t < t_final - 1e-10:
        # RK4 para sistema 2D
        k1y = v
        k1v = f(t, y, v)

        k2y = v + 0.5 * h * k1v
        k2v = f(t + 0.5*h, y + 0.5*h*k1y, v + 0.5*h*k1v)

        k3y = v + 0.5 * h * k2v
        k3v = f(t + 0.5*h, y + 0.5*h*k2y, v + 0.5*h*k2v)

        k4y = v + h * k3v
        k4v = f(t + h, y + h*k3y, v + h*k3v)

        y = y + (h / 6) * (k1y + 2*k2y + 2*k3y + k4y)
        v = v + (h / 6) * (k1v + 2*k2v + 2*k3v + k4v)
        t = round(t + h, 10)

        ts.append(t); ys.append(y); vs.append(v)

    print(f"[2ª Ordem] Convertida em sistema 2×1 | Passos: {len(ts)-1}")
    return ts, ys, vs


# ─────────────────────────────────────────────────────────────────────────────
# 4. LINEARIDADE DE EQUAÇÕES DIFERENCIAIS
# ─────────────────────────────────────────────────────────────────────────────

def classificar_linearidade(descricao: str, eh_linear: bool,
                             justificativa: str) -> dict:
    """
    Classifica e explica a linearidade de uma equação diferencial.

    Uma EDO é LINEAR se puder ser escrita na forma:
        aₙ(x)·y⁽ⁿ⁾ + ... + a₁(x)·y' + a₀(x)·y = g(x)
    onde os coeficientes aᵢ(x) dependem APENAS de x (não de y).

    É NÃO LINEAR se y ou suas derivadas aparecem:
        - Elevadas a potências (y², y'³)
        - Em funções trigonométricas/exponenciais (sin(y), eʸ)
        - Multiplicadas entre si (y·y')

    Parâmetros:
        descricao    : Descrição da equação
        eh_linear    : True se linear, False se não linear
        justificativa: Explicação da classificação

    Retorna:
        dict com a classificação completa
    """
    classificacao = {
        "equacao": descricao,
        "tipo": "Linear" if eh_linear else "Não Linear",
        "justificativa": justificativa
    }
    print(f"\n{'─'*55}")
    print(f"  Equação     : {descricao}")
    print(f"  Linearidade : {classificacao['tipo']}")
    print(f"  Motivo      : {justificativa}")
    print(f"{'─'*55}")
    return classificacao


# ─────────────────────────────────────────────────────────────────────────────
# 5. MÉTODO DE EULER (RETA TANGENTE)
# ─────────────────────────────────────────────────────────────────────────────

def euler_passo(f: Callable[[float, float], float],
                x: float, y: float, h: float) -> float:
    """
    Calcula UM passo do Método de Euler (reta tangente).

    Fórmula:
        y_{n+1} = y_n + h · f(x_n, y_n)

    O método usa a inclinação no ponto ATUAL para avançar.
    Erro local: O(h²) | Erro global: O(h) → método de 1ª ordem.

    Parâmetros:
        f : Função f(x, y) = dy/dx
        x : Ponto atual
        y : Valor atual
        h : Tamanho do passo

    Retorna:
        y no próximo passo
    """
    return y + h * f(x, y)


def metodo_euler(f: Callable[[float, float], float],
                 x0: float, y0: float,
                 x_final: float, h: float) -> Tuple[List[float], List[float]]:
    """
    Aplica o Método de Euler no intervalo completo [x0, x_final].

    Parâmetros:
        f       : Função f(x, y) = dy/dx
        x0      : Valor inicial de x
        y0      : Condição inicial y(x0)
        x_final : Limite superior
        h       : Tamanho do passo

    Retorna:
        (xs, ys) : Valores de x e y calculados
    """
    xs, ys = [x0], [y0]
    x, y = x0, y0

    while x < x_final - 1e-10:
        y = euler_passo(f, x, y, h)
        x = round(x + h, 10)
        xs.append(x); ys.append(y)

    print(f"[Euler] Passos: {len(xs)-1} | h = {h:.4f} | y_final ≈ {ys[-1]:.6f}")
    return xs, ys


# ─────────────────────────────────────────────────────────────────────────────
# 6. MÉTODO DE HEUN (EULER MODIFICADO / PREDITOR-CORRETOR)
# ─────────────────────────────────────────────────────────────────────────────

def heun_passo(f: Callable[[float, float], float],
               x: float, y: float, h: float) -> float:
    """
    Calcula UM passo do Método de Heun (Euler modificado).

    Fórmula (preditor-corretor):
        Preditor:  ỹ_{n+1} = y_n + h · f(x_n, y_n)
        Corretor:  y_{n+1} = y_n + (h/2) · [f(x_n, y_n) + f(x_{n+1}, ỹ_{n+1})]

    Usa a média das inclinações no início e no fim do passo.
    Erro local: O(h³) | Erro global: O(h²) → método de 2ª ordem.

    Parâmetros:
        f : Função f(x, y) = dy/dx
        x : Ponto atual
        y : Valor atual
        h : Tamanho do passo

    Retorna:
        y no próximo passo
    """
    k1 = f(x, y)                       # Inclinação no início
    y_pred = y + h * k1                 # Preditor (Euler simples)
    k2 = f(x + h, y_pred)              # Inclinação no fim (previsto)
    return y + (h / 2) * (k1 + k2)     # Corretor (média das inclinações)


def metodo_heun(f: Callable[[float, float], float],
                x0: float, y0: float,
                x_final: float, h: float) -> Tuple[List[float], List[float]]:
    """
    Aplica o Método de Heun no intervalo completo [x0, x_final].

    Parâmetros:
        f       : Função f(x, y) = dy/dx
        x0      : Valor inicial de x
        y0      : Condição inicial y(x0)
        x_final : Limite superior
        h       : Tamanho do passo

    Retorna:
        (xs, ys) : Valores de x e y calculados
    """
    xs, ys = [x0], [y0]
    x, y = x0, y0

    while x < x_final - 1e-10:
        y = heun_passo(f, x, y, h)
        x = round(x + h, 10)
        xs.append(x); ys.append(y)

    print(f"[Heun]  Passos: {len(xs)-1} | h = {h:.4f} | y_final ≈ {ys[-1]:.6f}")
    return xs, ys


# ─────────────────────────────────────────────────────────────────────────────
# 7. MÉTODO DE RUNGE-KUTTA (GENÉRICO — TABLEAU DE BUTCHER)
# ─────────────────────────────────────────────────────────────────────────────

def runge_kutta_generico(f: Callable[[float, float], float],
                         x0: float, y0: float,
                         x_final: float, h: float,
                         c: List[float],
                         a: List[List[float]],
                         b: List[float]) -> Tuple[List[float], List[float]]:
    """
    Método de Runge-Kutta genérico definido pelo Tableau de Butcher.

    Tableau de Butcher:
        c | A
        ──┼──
          | bᵀ

    Onde:
        c : nós (c_i)
        a : coeficientes (a_{ij}) — matriz triangular inferior
        b : pesos (b_i) com Σb_i = 1

    Estágios:
        kᵢ = f(x + cᵢ·h, y + h·Σⱼ aᵢⱼ·kⱼ)
        y_{n+1} = y_n + h·Σᵢ bᵢ·kᵢ

    Parâmetros:
        f       : Função f(x, y) = dy/dx
        x0      : Valor inicial de x
        y0      : Condição inicial
        x_final : Limite superior
        h       : Tamanho do passo
        c       : Lista de nós cᵢ
        a       : Matriz de coeficientes aᵢⱼ
        b       : Lista de pesos bᵢ

    Retorna:
        (xs, ys) : Valores calculados
    """
    s = len(b)           # Número de estágios
    xs, ys = [x0], [y0]
    x, y = x0, y0

    while x < x_final - 1e-10:
        k = [0.0] * s
        for i in range(s):
            soma = sum(a[i][j] * k[j] for j in range(i))
            k[i] = f(x + c[i] * h, y + h * soma)
        y = y + h * sum(b[i] * k[i] for i in range(s))
        x = round(x + h, 10)
        xs.append(x); ys.append(y)

    print(f"[RK Genérico] {s} estágios | Passos: {len(xs)-1} | y_final ≈ {ys[-1]:.6f}")
    return xs, ys


# ─────────────────────────────────────────────────────────────────────────────
# 8. MÉTODO DE RUNGE-KUTTA DE 3ª ORDEM (RK3) — DUAS VARIANTES
# ─────────────────────────────────────────────────────────────────────────────

def rk3_classico_passo(f: Callable[[float, float], float],
                       x: float, y: float, h: float) -> float:
    """
    Calcula UM passo do RK3 — Variante Clássica (Simpson).

    Tableau de Butcher:
        0   |
        1/2 | 1/2
        1   | -1    2
        ────┼──────────────
            | 1/6  4/6  1/6

    Fórmula:
        k₁ = f(x,       y)
        k₂ = f(x + h/2, y + h·k₁/2)
        k₃ = f(x + h,   y - h·k₁ + 2h·k₂)
        y_{n+1} = y_n + (h/6)·(k₁ + 4k₂ + k₃)

    Erro local: O(h⁴) | Erro global: O(h³).
    """
    k1 = f(x,         y)
    k2 = f(x + h/2,   y + h * k1 / 2)
    k3 = f(x + h,     y - h * k1 + 2 * h * k2)
    return y + (h / 6) * (k1 + 4 * k2 + k3)


def rk3_ralston_passo(f: Callable[[float, float], float],
                      x: float, y: float, h: float) -> float:
    """
    Calcula UM passo do RK3 — Variante de Ralston.

    Tableau de Butcher:
        0    |
        1/2  | 1/2
        3/4  | 0     3/4
        ─────┼──────────────
             | 2/9  3/9  4/9

    Fórmula:
        k₁ = f(x,          y)
        k₂ = f(x + h/2,    y + h·k₁/2)
        k₃ = f(x + 3h/4,   y + 3h·k₂/4)
        y_{n+1} = y_n + (h/9)·(2k₁ + 3k₂ + 4k₃)

    Erro local: O(h⁴) | Erro global: O(h³).
    Minimiza o erro de truncamento local entre as variantes de RK3.
    """
    k1 = f(x,           y)
    k2 = f(x + h/2,     y + h * k1 / 2)
    k3 = f(x + 3*h/4,   y + 3 * h * k2 / 4)
    return y + (h / 9) * (2 * k1 + 3 * k2 + 4 * k3)


# Alias padrão — aponta para Clássica (pode ser redirecionado)
def rk3_passo(f, x, y, h):
    """Alias para rk3_classico_passo (variante padrão)."""
    return rk3_classico_passo(f, x, y, h)


def metodo_rk3(f: Callable[[float, float], float],
               x0: float, y0: float,
               x_final: float, h: float,
               variante: str = "classica") -> Tuple[List[float], List[float]]:
    """
    Aplica o Método de Runge-Kutta de 3ª Ordem no intervalo [x0, x_final].

    Parâmetros:
        f        : Função f(x, y) = dy/dx
        x0       : Valor inicial de x
        y0       : Condição inicial y(x0)
        x_final  : Limite superior
        h        : Tamanho do passo
        variante : 'classica' (Simpson) ou 'ralston'

    Retorna:
        (xs, ys) : Valores calculados
    """
    if variante == "ralston":
        passo_fn = rk3_ralston_passo
    elif variante == "classica":
        passo_fn = rk3_classico_passo
    else:
        raise ValueError(f"Variante '{variante}' inválida. Use 'classica' ou 'ralston'.")

    xs, ys = [x0], [y0]
    x, y = x0, y0

    while x < x_final - 1e-10:
        y = passo_fn(f, x, y, h)
        x = round(x + h, 10)
        xs.append(x)
        ys.append(y)

    print(f"[RK3-{variante.capitalize():<8}] Passos: {len(xs)-1} | h = {h:.4f} | y_final ≈ {ys[-1]:.6e}")
    return xs, ys


# ─────────────────────────────────────────────────────────────────────────────
# 9. MÉTODO DE RUNGE-KUTTA DE 4ª ORDEM (RK4) — CLÁSSICO
# ─────────────────────────────────────────────────────────────────────────────

def rk4_passo(f: Callable[[float, float], float],
              x: float, y: float, h: float) -> float:
    """
    Calcula UM passo do Método de Runge-Kutta de 4ª Ordem (RK4 clássico).

    Fórmula:
        k₁ = f(x,         y)
        k₂ = f(x + h/2,   y + h·k₁/2)
        k₃ = f(x + h/2,   y + h·k₂/2)
        k₄ = f(x + h,     y + h·k₃)
        y_{n+1} = y_n + (h/6)·(k₁ + 2k₂ + 2k₃ + k₄)

    Interpretação dos estágios:
        k₁ : Inclinação no início do intervalo
        k₂ : Inclinação no meio (usando k₁)
        k₃ : Inclinação no meio (usando k₂)
        k₄ : Inclinação no fim do intervalo
    A média ponderada (1, 2, 2, 1) dá mais peso ao meio do intervalo.

    Erro local: O(h⁵) | Erro global: O(h⁴) → método de 4ª ordem.
    Requer 4 avaliações de f por passo.

    Parâmetros:
        f : Função f(x, y) = dy/dx
        x : Ponto atual
        y : Valor atual
        h : Tamanho do passo

    Retorna:
        y no próximo passo
    """
    k1 = f(x,           y)
    k2 = f(x + h / 2,   y + h * k1 / 2)
    k3 = f(x + h / 2,   y + h * k2 / 2)
    k4 = f(x + h,       y + h * k3)
    return y + (h / 6) * (k1 + 2*k2 + 2*k3 + k4)


def metodo_rk4(f: Callable[[float, float], float],
               x0: float, y0: float,
               x_final: float, h: float) -> Tuple[List[float], List[float]]:
    """
    Aplica o Método de Runge-Kutta de 4ª Ordem no intervalo [x0, x_final].

    Parâmetros:
        f       : Função f(x, y) = dy/dx
        x0      : Valor inicial de x
        y0      : Condição inicial y(x0)
        x_final : Limite superior
        h       : Tamanho do passo

    Retorna:
        (xs, ys) : Valores calculados
    """
    xs, ys = [x0], [y0]
    x, y = x0, y0

    while x < x_final - 1e-10:
        y = rk4_passo(f, x, y, h)
        x = round(x + h, 10)
        xs.append(x); ys.append(y)

    print(f"[RK4]   Passos: {len(xs)-1} | h = {h:.4f} | y_final ≈ {ys[-1]:.6f}")
    return xs, ys


# ─────────────────────────────────────────────────────────────────────────────
# COMPARAÇÃO DE ERROS ENTRE MÉTODOS
# ─────────────────────────────────────────────────────────────────────────────

def comparar_metodos(f: Callable[[float, float], float],
                     solucao_exata: Callable[[float], float],
                     x0: float, y0: float,
                     x_final: float, h: float) -> dict:
    """
    Compara Euler, Heun, RK3 e RK4 contra a solução analítica exata.

    Parâmetros:
        f             : Função f(x, y) = dy/dx
        solucao_exata : Função analítica y(x)
        x0, y0        : Condição inicial
        x_final       : Limite superior
        h             : Tamanho do passo

    Retorna:
        dict com resultados e erros de cada método
    """
    resultados = {}

    metodos = {
        "Euler": metodo_euler,
        "Heun":  metodo_heun,
        "RK3":   metodo_rk3,
        "RK4":   metodo_rk4,
    }

    x_exato = np.linspace(x0, x_final, 500)
    y_exato = [solucao_exata(xi) for xi in x_exato]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Comparação de Métodos Numéricos", fontsize=14, fontweight="bold")

    cores = {"Euler": "#e74c3c", "Heun": "#f39c12", "RK3": "#2ecc71", "RK4": "#3498db"}

    axes[0].plot(x_exato, y_exato, "k--", lw=2, label="Solução Exata", zorder=5)

    for nome, metodo_fn in metodos.items():
        xs, ys = metodo_fn(f, x0, y0, x_final, h)
        y_ref = solucao_exata(xs[-1])
        erro = abs(ys[-1] - y_ref)
        resultados[nome] = {"xs": xs, "ys": ys, "erro_final": erro}

        axes[0].plot(xs, ys, "-o", ms=3, color=cores[nome], label=nome, alpha=0.85)

        # Gráfico de erro absoluto
        erros = [abs(y - solucao_exata(x)) for x, y in zip(xs, ys)]
        axes[1].semilogy(xs, erros, "-", color=cores[nome], label=nome, lw=1.8)

    axes[0].set_title("Solução Numérica vs. Exata"); axes[0].legend(); axes[0].grid(True, alpha=0.3)
    axes[0].set_xlabel("x"); axes[0].set_ylabel("y")
    axes[1].set_title("Erro Absoluto (escala log)"); axes[1].legend(); axes[1].grid(True, alpha=0.3)
    axes[1].set_xlabel("x"); axes[1].set_ylabel("|erro|")

    plt.tight_layout()
    plt.savefig("comparacao_metodos.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("\n[Gráfico] Salvo em comparacao_metodos.png")

    print(f"\n{'─'*55}")
    print(f"  {'Método':<10} {'y_final numérico':>18} {'Erro absoluto':>16}")
    print(f"{'─'*55}")
    for nome, r in resultados.items():
        print(f"  {nome:<10} {r['ys'][-1]:>18.8f} {r['erro_final']:>16.2e}")
    print(f"  {'Exato':<10} {solucao_exata(x_final):>18.8f}")
    print(f"{'─'*55}")

    return resultados


# ─────────────────────────────────────────────────────────────────────────────
# DEMONSTRAÇÃO COMPLETA
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    print("=" * 55)
    print("   CÁLCULO NUMÉRICO — EQUAÇÕES DIFERENCIAIS")
    print("=" * 55)

    # ── Problema de demonstração ──────────────────────────────
    # EDO: dy/dx = -2y + 4x,  y(0) = 1
    # Solução exata: y(x) = 2x - 1 + 2·e^(-2x)

    def f(x, y):
        return (10 * y) * math.cos(x)  #-2 * y + 4 * x

    def y_exato(x):
        return 2*x - 1 + 2 * np.exp(-2 * x)

    x0, y0 = 0.0, 1.0
    x_final = 2.0
    h = 0.1

    # 1. EDO
    print("\n1. EQUAÇÃO DIFERENCIAL ORDINÁRIA")
    edo_exemplo(f, x0, y0, x_final, h, metodo="rk4")

    # 2. EDP — Equação do Calor
    print("\n2. EQUAÇÃO DIFERENCIAL PARCIAL (Calor)")
    x_grid, t_grid, u = edp_equacao_calor(
        L=1.0, T=0.1, nx=51, nt=1000, alpha=0.01,
        u_inicial=lambda x: np.sin(np.pi * x)
    )
    print(f"   u no centro (x=0.5) em t=T: {u[-1, 25]:.6f}")

    # 3. Ordem
    print("\n3. ORDEM DAS EQUAÇÕES DIFERENCIAIS")
    verificar_ordem("dy/dx = -2y + 4x", 1)
    verificar_ordem("d²y/dx² + 4y = 0", 2)
    verificar_ordem("d³y/dx³ = cos(x)", 3)

    # 4. Linearidade
    print("\n4. LINEARIDADE")
    classificar_linearidade(
        "dy/dx = -2y + 4x", True,
        "Coeficientes de y dependem apenas de x — linear"
    )
    classificar_linearidade(
        "dy/dx = y² + x", False,
        "y aparece elevado ao quadrado (y²) — não linear"
    )
    classificar_linearidade(
        "y'' + sin(y) = 0", False,
        "sin(y) é não linear em relação a y"
    )

    # 5. Euler
    print("\n5. MÉTODO DE EULER")
    metodo_euler(f, x0, y0, x_final, h)

    # 6. Heun
    print("\n6. MÉTODO DE HEUN")
    metodo_heun(f, x0, y0, x_final, h)

    # 7. RK Genérico (usando tableau do RK4)
    print("\n7. RUNGE-KUTTA GENÉRICO (Tableau RK4)")
    c_rk4 = [0, 1/2, 1/2, 1]
    a_rk4 = [[0,   0,   0, 0],
              [1/2, 0,   0, 0],
              [0,   1/2, 0, 0],
              [0,   0,   1, 0]]
    b_rk4 = [1/6, 1/3, 1/3, 1/6]
    runge_kutta_generico(f, x0, y0, x_final, h, c_rk4, a_rk4, b_rk4)

    # 8. RK3 — duas variantes
    print("\n8. RUNGE-KUTTA 3ª ORDEM — Classica vs Ralston")
    metodo_rk3(f, x0, y0, x_final, h, variante="classica")
    metodo_rk3(f, x0, y0, x_final, h, variante="ralston")

    # Teste populacional
    print("\n── TESTE: Crescimento Populacional (dy/dx = 10y*cos(x)) ──")
    import math
    def f_pop(x, y): return 10 * y * math.cos(x)
    print(f"  {'Passo':<6} {'x':<5} {'Ano':<8} {'Classica':>22} {'Ralston':>22}")
    print("  " + "-" * 65)
    x, yc, yr = 0.0, 600.0, 600.0
    print(f"  {0:<6} {x:<5.1f} {int(2001+x):<8} {yc:>22.4f} {yr:>22.4f}")
    for i in range(18):
        yc = rk3_classico_passo(f_pop, x, yc, 0.5)
        yr = rk3_ralston_passo(f_pop, x, yr, 0.5)
        x = round(x + 0.5, 10)
        print(f"  {i+1:<6} {x:<5.1f} {int(2001+x):<8} {yc:>22.4f} {yr:>22.4f}")
    print(f"\n  Resultado esperado (Ralston): 4729326129300760.0")
    print(f"  Resultado obtido   (Ralston): {yr:.1f}")

    # 9. RK4
    print("\n9. RUNGE-KUTTA 4ª ORDEM")
    metodo_rk4(f, x0, y0, x_final, h)

    # Comparação completa com gráfico
    print("\n── COMPARAÇÃO COMPLETA ──")
    comparar_metodos(f, y_exato, x0, y0, x_final, h)

    print("\n✓ Demonstração concluída!")

   CÁLCULO NUMÉRICO — EQUAÇÕES DIFERENCIAIS

1. EQUAÇÃO DIFERENCIAL ORDINÁRIA
[EDO] Método 'rk4' | Passos: 20 | h = 0.1

2. EQUAÇÃO DIFERENCIAL PARCIAL (Calor)
[EDP] Equação do Calor | r = 0.0025 (estável) | nx=51, nt=1000
   u no centro (x=0.5) em t=T: 0.990182

3. ORDEM DAS EQUAÇÕES DIFERENCIAIS

───────────────────────────────────────────────────────
  Equação : dy/dx = -2y + 4x
  Ordem   : 1ª
  Tipo    : Primeira ordem — envolve dy/dx
───────────────────────────────────────────────────────

───────────────────────────────────────────────────────
  Equação : d²y/dx² + 4y = 0
  Ordem   : 2ª
  Tipo    : Segunda ordem  — envolve d²y/dx²
───────────────────────────────────────────────────────

───────────────────────────────────────────────────────
  Equação : d³y/dx³ = cos(x)
  Ordem   : 3ª
  Tipo    : 3ª ordem — maior derivada é d^3y/dx^3
───────────────────────────────────────────────────────

4. LINEARIDADE

───────────────────────────────────────────────────────
  Equação     : dy/